# Human Gene Essentiality Predictor V2 (Fixed)
## Proper Gene Name Matching Between CRISPR and Expression Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

In [ ]:
# Data paths
DATA_DIR = '/content/drive/MyDrive/depmap_data'

# List files in directory
print("Files in data directory:")
for f in os.listdir(DATA_DIR):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

In [ ]:
# Load CRISPR data
print("\n" + "="*60)
print("LOADING CRISPR DATA")
print("="*60)

crispr_path = os.path.join(DATA_DIR, 'CRISPRGeneEffect.csv')
crispr_raw = pd.read_csv(crispr_path, index_col=0)
print(f"Shape: {crispr_raw.shape}")
print(f"\nFirst few column names (genes):")
print(list(crispr_raw.columns[:5]))
print(f"\nFirst few row names (cell lines):")
print(list(crispr_raw.index[:5]))

In [ ]:
# Load Expression data
print("\n" + "="*60)
print("LOADING EXPRESSION DATA")
print("="*60)

# Find expression file
expr_path = None
for f in os.listdir(DATA_DIR):
    if 'expression' in f.lower() or 'omics' in f.lower():
        if 'crispr' not in f.lower():
            expr_path = os.path.join(DATA_DIR, f)
            print(f"Found expression file: {f}")
            break

if expr_path:
    expr_raw = pd.read_csv(expr_path, index_col=0)
    print(f"Shape: {expr_raw.shape}")
    print(f"\nFirst few column names (genes):")
    print(list(expr_raw.columns[:5]))
    print(f"\nFirst few row names (cell lines):")
    print(list(expr_raw.index[:5]))
else:
    print("ERROR: Expression file not found!")

In [ ]:
# DEBUG: Understand gene name formats
print("\n" + "="*60)
print("DEBUGGING GENE NAME FORMATS")
print("="*60)

print("\nCRISPR gene name examples:")
for g in list(crispr_raw.columns[:10]):
    print(f"  '{g}'")

print("\nExpression gene name examples:")
for g in list(expr_raw.columns[:10]):
    print(f"  '{g}'")

In [ ]:
# Function to extract gene symbol from various formats
def extract_gene_symbol(name):
    """
    Extract gene symbol from formats like:
    - 'GENE (12345)' -> 'GENE'
    - 'GENE' -> 'GENE'
    - 'gene' -> 'GENE'
    """
    name = str(name)
    # Remove anything in parentheses
    if ' (' in name:
        name = name.split(' (')[0]
    return name.strip().upper()

# Create mapping from symbol to original name
crispr_symbol_to_orig = {extract_gene_symbol(g): g for g in crispr_raw.columns}
expr_symbol_to_orig = {extract_gene_symbol(g): g for g in expr_raw.columns}

print(f"Unique CRISPR gene symbols: {len(crispr_symbol_to_orig)}")
print(f"Unique Expression gene symbols: {len(expr_symbol_to_orig)}")

# Find common symbols
common_symbols = set(crispr_symbol_to_orig.keys()) & set(expr_symbol_to_orig.keys())
print(f"Common gene symbols: {len(common_symbols)}")

In [ ]:
# Find common cell lines
print("\n" + "="*60)
print("MATCHING CELL LINES")
print("="*60)

crispr_cells = set(crispr_raw.index)
expr_cells = set(expr_raw.index)

common_cells = list(crispr_cells & expr_cells)
print(f"CRISPR cell lines: {len(crispr_cells)}")
print(f"Expression cell lines: {len(expr_cells)}")
print(f"Common cell lines: {len(common_cells)}")

if len(common_cells) == 0:
    print("\nWARNING: No common cell lines found!")
    print("\nCRISPR cell line format examples:")
    print(list(crispr_raw.index[:5]))
    print("\nExpression cell line format examples:")
    print(list(expr_raw.index[:5]))

In [ ]:
# Create aligned dataframes using common symbols and cells
print("\n" + "="*60)
print("CREATING ALIGNED DATAFRAMES")
print("="*60)

if len(common_cells) > 0 and len(common_symbols) > 0:
    # Get original column names for common symbols
    crispr_cols = [crispr_symbol_to_orig[s] for s in common_symbols]
    expr_cols = [expr_symbol_to_orig[s] for s in common_symbols]
    
    # Subset dataframes
    crispr_aligned = crispr_raw.loc[common_cells, crispr_cols].copy()
    expr_aligned = expr_raw.loc[common_cells, expr_cols].copy()
    
    # Rename columns to common symbols for alignment
    crispr_aligned.columns = list(common_symbols)
    expr_aligned.columns = list(common_symbols)
    
    # Transpose: genes x cell_lines
    crispr_df = crispr_aligned.T
    expr_df = expr_aligned.T
    
    print(f"Aligned CRISPR shape: {crispr_df.shape} (genes x cell_lines)")
    print(f"Aligned Expression shape: {expr_df.shape} (genes x cell_lines)")
    
    # Verify alignment
    assert list(crispr_df.index) == list(expr_df.index), "Gene indices don't match!"
    assert list(crispr_df.columns) == list(expr_df.columns), "Cell line columns don't match!"
    print("\n✅ Dataframes aligned successfully!")
else:
    print("ERROR: Cannot align dataframes!")

In [ ]:
# Binarize CRISPR scores
THRESHOLD = -0.5
binary_mat = (crispr_df < THRESHOLD).astype(int)

print(f"Binary matrix shape: {binary_mat.shape}")
print(f"Essential calls: {binary_mat.sum().sum():,} / {binary_mat.size:,} ({100*binary_mat.sum().sum()/binary_mat.size:.1f}%)")

In [ ]:
# Gene classification
gene_consensus = binary_mat.mean(axis=1)

HIGH_THRESH = 0.9
LOW_THRESH = 0.1

pan_mask = gene_consensus >= HIGH_THRESH
non_mask = gene_consensus <= LOW_THRESH
context_mask = ~pan_mask & ~non_mask

print(f"Pan-essential (≥{HIGH_THRESH:.0%}): {pan_mask.sum()} genes")
print(f"Non-essential (≤{LOW_THRESH:.0%}): {non_mask.sum()} genes")
print(f"Context-dependent: {context_mask.sum()} genes")

# List some context-dependent genes
context_genes = gene_consensus[context_mask].index.tolist()
print(f"\nExample context-dependent genes: {context_genes[:10]}")

## V2 Predictor

In [ ]:
class HedgeFundV2:
    def __init__(self, high_thresh=0.9, low_thresh=0.1):
        self.high_thresh = high_thresh
        self.low_thresh = low_thresh
        self.ml_model = GradientBoostingClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1, subsample=0.8
        )
        self.scaler = StandardScaler()
        self.is_fitted = False
        
    def _get_features(self, binary_mat, expr_mat, gene, cell, other_cells):
        # Consensus
        consensus = binary_mat.loc[gene, other_cells].mean()
        variance = binary_mat.loc[gene, other_cells].var()
        
        # Expression
        expr_val = expr_mat.loc[gene, cell]
        expr_mean = expr_mat.loc[gene, other_cells].mean()
        expr_std = expr_mat.loc[gene, other_cells].std() + 1e-6
        expr_z = (expr_val - expr_mean) / expr_std
        expr_pct = (expr_mat.loc[gene, :] < expr_val).mean()
        
        return [consensus, variance, expr_val, expr_z, expr_pct, 
                consensus * expr_val, consensus**2, expr_val**2]
    
    def fit(self, binary_mat, expr_mat, n_train=100):
        print("Training V2...")
        
        # Get context genes
        consensus = binary_mat.mean(axis=1)
        context_genes = consensus[(consensus > self.low_thresh) & 
                                   (consensus < self.high_thresh)].index.tolist()
        print(f"  Context genes: {len(context_genes)}")
        
        if len(context_genes) == 0:
            print("  ERROR: No context-dependent genes!")
            return self
        
        all_cells = list(binary_mat.columns)
        train_cells = all_cells[:n_train]
        print(f"  Training cells: {len(train_cells)}")
        
        X, y = [], []
        for cell in tqdm(train_cells, desc="  Building data"):
            other = [c for c in all_cells if c != cell]
            for gene in context_genes:
                try:
                    feat = self._get_features(binary_mat, expr_mat, gene, cell, other)
                    if not np.isnan(feat).any():
                        X.append(feat)
                        y.append(binary_mat.loc[gene, cell])
                except:
                    pass
        
        X, y = np.array(X), np.array(y)
        print(f"  Training samples: {len(X):,}")
        print(f"  Positive rate: {y.mean():.1%}")
        
        if len(X) == 0:
            print("  ERROR: No training samples!")
            return self
        
        X_scaled = self.scaler.fit_transform(X)
        self.ml_model.fit(X_scaled, y)
        
        y_pred = self.ml_model.predict(X_scaled)
        print(f"  Train accuracy: {accuracy_score(y, y_pred):.3f}")
        print(f"  Train balanced: {balanced_accuracy_score(y, y_pred):.3f}")
        
        # Feature importance
        names = ['consensus', 'variance', 'expr', 'expr_z', 'expr_pct', 
                 'cons*expr', 'cons^2', 'expr^2']
        print("\n  Feature importance:")
        for n, i in sorted(zip(names, self.ml_model.feature_importances_), key=lambda x: -x[1]):
            print(f"    {n:12s}: {i:.3f}")
        
        self.is_fitted = True
        return self
    
    def predict(self, binary_mat, expr_mat, cell):
        all_cells = list(binary_mat.columns)
        other = [c for c in all_cells if c != cell]
        
        consensus = binary_mat[other].mean(axis=1)
        preds = pd.Series(0, index=binary_mat.index)
        
        # Pan-essential
        preds[consensus >= self.high_thresh] = 1
        
        # Context-dependent: ML
        ctx_mask = (consensus > self.low_thresh) & (consensus < self.high_thresh)
        ctx_genes = preds[ctx_mask].index.tolist()
        
        if ctx_genes and self.is_fitted:
            X_test, valid = [], []
            for g in ctx_genes:
                try:
                    f = self._get_features(binary_mat, expr_mat, g, cell, other)
                    if not np.isnan(f).any():
                        X_test.append(f)
                        valid.append(g)
                except:
                    pass
            if X_test:
                X_test = self.scaler.transform(np.array(X_test))
                ml_pred = self.ml_model.predict(X_test)
                for g, p in zip(valid, ml_pred):
                    preds[g] = p
        
        return preds

print("HedgeFundV2 defined!")

In [ ]:
# Train
predictor = HedgeFundV2()
predictor.fit(binary_mat, expr_df, n_train=100)

In [ ]:
# Evaluate
if predictor.is_fitted:
    N_TEST = 50
    all_cells = list(binary_mat.columns)
    test_cells = all_cells[100:100+N_TEST]
    
    y_true_all, y_pred_all = [], []
    
    print(f"Evaluating on {len(test_cells)} cell lines...")
    for cell in tqdm(test_cells):
        y_true = binary_mat[cell].values
        y_pred = predictor.predict(binary_mat, expr_df, cell).values
        y_true_all.extend(y_true)
        y_pred_all.extend(y_pred)
    
    y_true = np.array(y_true_all)
    y_pred = np.array(y_pred_all)
    
    print("\n" + "="*60)
    print("V2 RESULTS")
    print("="*60)
    print(f"Accuracy:          {accuracy_score(y_true, y_pred):.3f}")
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.3f}")
    print(f"F1 Score:          {f1_score(y_true, y_pred):.3f}")

In [ ]:
# Accuracy by category
if predictor.is_fitted:
    categories = {
        'Pan-essential (≥90%)': gene_consensus >= 0.9,
        'Common (50-90%)': (gene_consensus >= 0.5) & (gene_consensus < 0.9),
        'Context-dep (10-50%)': (gene_consensus >= 0.1) & (gene_consensus < 0.5),
        'Rarely ess (0-10%)': (gene_consensus > 0) & (gene_consensus < 0.1),
        'Non-essential (0%)': gene_consensus == 0
    }
    
    v1_results = {
        'Pan-essential (≥90%)': 0.982,
        'Common (50-90%)': 0.434,
        'Context-dep (10-50%)': 0.439,
        'Rarely ess (0-10%)': 0.991,
        'Non-essential (0%)': 1.000
    }
    
    print("\nV1 → V2 Comparison:")
    print("-" * 70)
    
    for cat, mask in categories.items():
        n = mask.sum()
        if n > 0:
            mask_exp = np.tile(mask.values, N_TEST)
            if mask_exp.sum() > 0:
                v2_acc = accuracy_score(y_true[mask_exp], y_pred[mask_exp])
                v1_acc = v1_results.get(cat, 0)
                diff = v2_acc - v1_acc
                emoji = "✅" if diff > 0.05 else "➡️" if diff > -0.05 else "❌"
                print(f"{cat:25s}: {v1_acc:.1%} → {v2_acc:.1%} ({diff:+.1%}) {emoji} (n={n})")

In [ ]:
# Plot
if predictor.is_fitted:
    cats = list(categories.keys())
    v1_accs = [v1_results[c] for c in cats]
    v2_accs = []
    for cat, mask in categories.items():
        mask_exp = np.tile(mask.values, N_TEST)
        if mask_exp.sum() > 0:
            v2_accs.append(accuracy_score(y_true[mask_exp], y_pred[mask_exp]))
        else:
            v2_accs.append(0)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(cats))
    w = 0.35
    
    ax.bar(x - w/2, v1_accs, w, label='V1 (Consensus)', color='#ff6b6b', alpha=0.8)
    ax.bar(x + w/2, v2_accs, w, label='V2 (+Expression)', color='#4ecdc4', alpha=0.8)
    ax.axhline(0.5, color='gray', ls='--', alpha=0.5)
    ax.set_ylabel('Accuracy')
    ax.set_title('V1 vs V2: Accuracy by Gene Category')
    ax.set_xticks(x)
    ax.set_xticklabels(cats, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    plt.show()